# 14 · Like for like: every technique on the same items

The LLM techniques (notebooks 03–08 and 11) were scored on fixed samples of test items: 600 MultiEURLEX laws and 1,500
HumSet excerpts, exactly and tier by tier. The fine-tuned models and the combined method (notebooks 12–15) were scored
on the full test sets with the benchmarks' own measures. Those numbers cannot be set side by side.

This notebook scores every technique on the same sample items with the same measures:

- **LLM techniques:** their saved labels are rescored, unchanged. Each must reproduce the score its notebook saved, or
  this notebook raises.
- **Ranked methods** (every trained or fine-tuned model, and the combined method): each gets a label limit and a
  score cut-off chosen on the same development items, the ones the combined method chose its cut-off on, which no
  model trained on: 500 later development laws and 4,000 development excerpts. The choices are the same as notebook
  10's.
- **The combined method** is retrained exactly as notebooks 10 and 11 trained it, on the same development items.

**No pass mark.** This notebook only rescores; nothing here is chosen on test items. The test items were already
scored by every method, so these are not new test results; they put the earlier ones on one scale.

Two earlier rows are left out because a row below replaces them: the bake-off's "trained" model is Omikuji with only a
label limit, and its "retrieval" is label search with a fixed ten labels. Both appear here with a tuned cut-off.
Notebook 12's first combined method was only run on HumSet and is replaced by notebooks 10 and 11.

In [1]:
from collections import Counter

import numpy as np
import pandas as pd

from tiered_taxonomy import benchmarks as bm
from tiered_taxonomy import experiment as ex
from tiered_taxonomy import like_for_like as lf
from tiered_taxonomy import new_technique as nt
from tiered_taxonomy import signals as sg
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.methods import combined as cb

root = data_root()
config = load_config()
seed = config["seed"]
words = config["text"]["input_words"]
samples = config["samples"]
limits, thresholds = config["trained"]["cut_limits"], config["trained"]["cut_thresholds"]
settings = cb.CombinedSettings(per_source=config["combined"]["per_source"], seed=seed)
plan = config["new_technique"]
recent = config["second_attempt"]["recent_documents"]
first = ex.load_results("10_combined_method")
second = ex.load_results("11_second_attempt")
SAVED = {name: ex.load_results(name) for name in ("03_bakeoff", "04_funnel", "05_tree_walk", "06_evidence")}
PAPER = ex.load_results("09_paper_recipes")
LEFT_OUT = {"trained", "retrieval"}
humset = ex.load_humset(root)
eurlex = ex.load_eurlex(root)
results = {}

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The samples and the development items

The same samples as every earlier notebook, and the development items the combined method used.

In [2]:
e_sample = ex.sample_eurlex(
    eurlex,
    samples["eurlex"]["normal_documents"],
    samples["eurlex"]["long_documents"],
    samples["eurlex"]["long_threshold_words"],
    seed,
)
h_sample = ex.sample_humset(humset, samples["humset"]["per_language"], seed)

order = np.random.default_rng(seed).permutation(len(humset.dev))
h_raw = [humset.dev[int(i)] for i in order[: plan["humset"]["fit_items"] + plan["humset"]["cut_items"]]]
h_fit = nt.with_all_tiers(h_raw[: plan["humset"]["fit_items"]])
h_cut = nt.with_all_tiers(h_raw[plan["humset"]["fit_items"] :])
h_tune = h_raw[plan["humset"]["fit_items"] :]

early, late = bm.split_by_date(eurlex.dev)
late = late[: plan["eurlex"]["fit_items"] + plan["eurlex"]["cut_items"]]
e_fit, e_cut = late[: plan["eurlex"]["fit_items"]], late[plan["eurlex"]["fit_items"] :]
pd.DataFrame(
    {
        "sample": {"MultiEURLEX": len(e_sample), "HumSet": len(h_sample)},
        "cut-off chosen on": {"MultiEURLEX": len(e_cut), "HumSet": len(h_tune)},
    }
)

,sample,cut-off chosen on
MultiEURLEX,600,500
HumSet,1500,4000


## LLM techniques: their saved labels, rescored

In [3]:
def llm_rows(data, sample):
    ids = [item.id for item in sample]
    rows = {}
    saved = [
        (name, method, entry)
        for name, body in SAVED.items()
        for method, entry in body["results"][data.name].items()
    ]
    saved += [
        ("09_paper_recipes", method, PAPER[data.name][method]) for method in ("ki_fsprompt", "llm_ensemble")
    ]
    for notebook, method, entry in saved:
        if method in LEFT_OUT or method in rows:
            continue
        scores = ex.evaluate(data, sample, ex.load_predictions(root, data.name, method, ids))
        lf.check_reproduces(method, scores, entry["scores"])
        run = entry["run"]
        rows[method] = {
            "family": "llm",
            "notebook": notebook,
            "scores": scores,
            "cost_per_1000": round(1000 * run["cost_usd"] / run["items"], 3),
            "calls_per_item": round(run["calls"] / run["items"], 2),
        }
    return rows


results["eurlex"] = {"llm": llm_rows(eurlex, e_sample)}
results["humset"] = {"llm": llm_rows(humset, h_sample)}
print(
    "every saved score reproduced:",
    len(results["eurlex"]["llm"]),
    "on MultiEURLEX,",
    len(results["humset"]["llm"]),
    "on HumSet",
)

every saved score reproduced: 11 on MultiEURLEX, 11 on HumSet


## Trained and fine-tuned models, each with a cut-off chosen on development items

On MultiEURLEX, "training laws only" models are the ones notebook 12 compared with BERT; "with newer laws" models also
trained on the earlier half of the development laws, by date.

In [4]:
def ranked_rows(data, sample, tune, sources, tag):
    rows = {}
    for name, rankings in sources.items():
        made = lf.score_ranked(
            data,
            tune,
            lf.source_rankings(rankings, tune, name),
            sample,
            lf.source_rankings(rankings, sample, name),
            limits,
            thresholds,
        )
        made.pop("labels")
        rows[f"{name}{tag}"] = {"family": "ranked", **made}
    return rows


E_TRAIN_ONLY = ["omikuji", "xr_transformer", "embedding_classifier", "knn", "label_search", "classifier"]
E_NEWER = [
    "classifier",
    "llm",
    "reranker",
    "gliner",
    "embedding_classifier",
    "knn",
    "label_search",
    "omikuji",
]
H_SOURCES = [
    "classifier",
    "llm",
    "reranker",
    "gliner",
    "xr_transformer",
    "omikuji",
    "embedding_classifier",
    "knn",
    "label_search",
    "humbert_recipe",
]
e_train_dev = sg.load_sources(root, eurlex.name, E_TRAIN_ONLY, "dev")
e_train_test = sg.load_sources(root, eurlex.name, E_TRAIN_ONLY, "test")
e_newer_dev = sg.load_sources(root, sg.NEWER_LAWS_SUFFIX.format("eurlex"), E_NEWER, "dev")
e_newer_test = sg.load_sources(root, sg.NEWER_LAWS_SUFFIX.format("eurlex"), E_NEWER, "test")
h_dev = sg.load_sources(root, humset.name, H_SOURCES, "dev")
h_test = sg.load_sources(root, humset.name, H_SOURCES, "test")


def merged(dev, test):
    return {name: {**dev[name], **test[name]} for name in dev}


results["eurlex"]["ranked"] = {
    **ranked_rows(eurlex, e_sample, e_cut, merged(e_train_dev, e_train_test), ":train_only"),
    **ranked_rows(eurlex, e_sample, e_cut, merged(e_newer_dev, e_newer_test), ":newer"),
}
results["humset"]["ranked"] = ranked_rows(humset, h_sample, h_tune, merged(h_dev, h_test), "")
pd.DataFrame(
    {
        name: {"limit": row["limit"], "cut-off": row["threshold"], "exact F1": row["scores"]["exact"]["f1"]}
        for key in ("eurlex", "humset")
        for name, row in {f"{key} {n}": r for n, r in results[key]["ranked"].items()}.items()
    }
).T.round(3)

,limit,cut-off,exact F1
eurlex omikuji:train_only,10.0,0.10,0.427
eurlex xr_transformer:train_only,10.0,0.15,0.434
eurlex embedding_classifier:train_only,6.0,0.10,0.416
eurlex knn:train_only,10.0,0.50,0.386
eurlex label_search:train_only,10.0,0.50,0.115
eurlex classifier:train_only,10.0,0.40,0.475
eurlex classifier:newer,15.0,0.35,0.497
eurlex llm:newer,15.0,0.40,0.461
eurlex reranker:newer,5.0,0.80,0.398
eurlex gliner:newer,15.0,0.35,0.470


## The combined method, retrained as the test runs trained it

In [5]:
def dated(items):
    return sorted(((bm.law_year(item.id), sorted(item.labels)) for item in items), key=lambda pair: pair[0])


def combined_row(stack, data, tune, sample, allowed=None):
    tune_rankings, sample_rankings = lf.combined_rankings(stack, sample)
    if allowed is not None:
        tune_rankings, sample_rankings = (
            lf.restrict(tune_rankings, allowed),
            lf.restrict(sample_rankings, allowed),
        )
    made = lf.score_ranked(data, tune, tune_rankings, sample, sample_rankings, limits, thresholds)
    made.pop("labels")
    return {
        "family": "combined",
        "sources": list(stack.names),
        "priors": stack.graph.priors is not None,
        **made,
    }


# The second attempt's stack is the final version. In our run it kept the first attempt's; a rerun may not,
# because Omikuji, one of the candidates, trains from random starting points that cannot be fixed.
h_names = second["humset"]["sources"]
h_graph = cb.LabelGraph(humset.taxonomy, [sorted(bm.gold_with_parents(item)) for item in humset.train])
h_dev_sources = nt.expand_sources(sg.load_sources(root, humset.name, h_names, "dev"))
h_test_sources = nt.expand_sources(sg.load_sources(root, humset.name, h_names, "test"))
h_stack = nt.Stack(h_names, h_fit, h_cut, h_dev_sources, h_test_sources, h_graph, words, settings)
exact_labels = {label for item in humset.train for label in item.labels}
results["humset"]["combined"] = {"combined": combined_row(h_stack, humset, h_tune, h_sample, exact_labels)}

newer_documents = [sorted(item.labels) for item in [*eurlex.train, *early]]
train_documents = [sorted(item.labels) for item in eurlex.train]
graphs = {
    "newer": cb.LabelGraph(eurlex.taxonomy, newer_documents),
    "newer_timed": cb.LabelGraph(
        eurlex.taxonomy,
        newer_documents,
        priors=cb.TimedPriors.from_documents(dated([*eurlex.train, *early]), recent),
    ),
    "train": cb.LabelGraph(eurlex.taxonomy, train_documents),
    "train_timed": cb.LabelGraph(
        eurlex.taxonomy, train_documents, priors=cb.TimedPriors.from_documents(dated(eurlex.train), recent)
    ),
}
runs = {
    "combined_first:newer": (first["eurlex"]["sources"], True, "newer"),
    "combined_second:newer": (
        second["eurlex"]["sources"],
        True,
        "newer_timed" if second["eurlex"]["priors"] else "newer",
    ),
    "combined_first:train_only": (first["eurlex_train_only"]["sources"], False, "train"),
    "combined_second:train_only": (
        second["eurlex_train_only"]["sources"],
        False,
        "train_timed" if second["eurlex_train_only"]["priors"] else "train",
    ),
}
results["eurlex"]["combined"] = {}
for key, (names, with_newer, graph) in runs.items():
    dev_sources = nt.eurlex_sources(root, names, "dev", with_newer=with_newer)
    test_sources = nt.eurlex_sources(root, names, "test", with_newer=with_newer)
    stack = nt.Stack(names, e_fit, e_cut, dev_sources, test_sources, graphs[graph], words, settings)
    results["eurlex"]["combined"][key] = combined_row(stack, eurlex, e_cut, e_sample)
pd.DataFrame(
    {
        f"{test_set} {name}": {
            "limit": row["limit"],
            "cut-off": row["threshold"],
            "exact F1": row["scores"]["exact"]["f1"],
        }
        for test_set in ("eurlex", "humset")
        for name, row in results[test_set]["combined"].items()
    }
).T.round(3)

,limit,cut-off,exact F1
eurlex combined_first:newer,10.0,0.40,0.549
eurlex combined_second:newer,10.0,0.40,0.566
eurlex combined_first:train_only,10.0,0.35,0.525
eurlex combined_second:train_only,8.0,0.25,0.551
humset combined,10.0,0.35,0.662


## Every technique on one scale

Exact-label F1 and F1 at each tier, on the same items. Best first.

In [6]:
def table(test_set):
    rows = {}
    for family in ("llm", "ranked", "combined"):
        for name, row in results[test_set][family].items():
            s = row["scores"]
            entry = {"family": family, "exact F1": s["exact"]["f1"]}
            entry.update({f"tier {t}": s["tiers"][t]["f1"] for t in sorted(s["tiers"])})
            entry.update(
                {f"{band} recall": value["recall"] for band, value in s["rarity"].items() if band != "common"}
            )
            entry["labels per item"] = s["labels_per_item"]
            rows[name] = entry
    return pd.DataFrame(rows).T.sort_values("exact F1", ascending=False)


table("eurlex").round(3)

,family,exact F1,tier 1,tier 2,tier 3,rare recall,unseen recall,labels per item
combined_second:newer,combined,0.566176,0.74967,0.709701,0.71253,0.1667,0.0083,3.91
combined_second:train_only,combined,0.550583,0.771443,0.721446,0.715342,0.086,0.0,5.06
combined_first:newer,combined,0.549412,0.733778,0.696931,0.698264,0.1183,0.0167,3.82
combined_first:train_only,combined,0.525272,0.731154,0.689831,0.68487,0.0484,0.0,3.9
classifier:newer,ranked,0.497093,0.748166,0.700187,0.688577,0.129,0.0083,5.24
classifier:train_only,ranked,0.474824,0.728423,0.675371,0.670522,0.0645,0.0,4.64
gliner:newer,ranked,0.469972,0.738863,0.681334,0.66952,0.1882,0.0583,4.61
omikuji:newer,ranked,0.462669,0.690759,0.646533,0.633191,0.0054,0.0,4.82
llm:newer,ranked,0.460646,0.703987,0.661897,0.657835,0.0161,0.0,4.11
embedding_classifier:newer,ranked,0.444444,0.709427,0.649141,0.637351,0.0108,0.0,5.48


In [7]:
table("humset").round(3)

,family,exact F1,tier 1,tier 2,rare recall,labels per item
combined,combined,0.661984,0.701907,0.584317,0.4355,2.39
xr_transformer,ranked,0.642313,0.685548,0.561629,0.4194,2.28
humbert_recipe,ranked,0.637095,0.699542,0.590807,0.4194,2.79
classifier,ranked,0.628289,0.700222,0.584834,0.5,2.77
llm,ranked,0.593538,0.675512,0.543256,0.4032,2.56
knn,ranked,0.589633,0.63105,0.507614,0.3065,2.14
omikuji,ranked,0.57934,0.617517,0.491927,0.3387,2.29
embedding_classifier,ranked,0.57306,0.611037,0.494905,0.3871,2.43
gliner,ranked,0.56533,0.653846,0.5055,0.3065,2.8
reranker,ranked,0.561209,0.619327,0.472887,0.3065,2.46


## Where the correct labels sit in the taxonomy

The exact score asks for the label a person assigned, at whatever tier they assigned it. The tier scores move
both the method's labels and the person's labels up to that tier first, so a near miss under the same parent
still counts there. This counts the tiers of the labels people assigned in each sample, which is why the
exact score and the tier scores differ.

In [8]:
def gold_tiers(data, sample):
    tiers = Counter(data.taxonomy.tiers[label] for item in sample for label in item.labels)
    return {"labels": sum(tiers.values()), "by_tier": {str(t): n for t, n in sorted(tiers.items())}}


for test_set, data, sample in (("eurlex", eurlex, e_sample), ("humset", humset, h_sample)):
    results[test_set]["gold"] = gold_tiers(data, sample)
pd.DataFrame({name: results[name]["gold"]["by_tier"] for name in ("eurlex", "humset")}).fillna(0).astype(int)

,eurlex,humset
3,241,0
4,2362,0
5,1018,0
6,281,0
7,9,0
1,0,1592
2,0,1937


In [9]:
ex.save_results("14_like_for_like", results)
print("saved")

saved
